# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimahmahmood/flyrank_ml_internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Abstract*

This study investigates whether search-performance and content-freshness signals can be used to identify content items that are likely to experience declining search performance. The analysis uses anonymized content-performance data and defines a declining item as one whose search impressions fall by more than 20% over the following 30-day period compared with the preceding period. A dataset of 331,436 content-client observations was constructed using a March 31, 2026 prediction snapshot and an April 2026 outcome period. A Decision Tree classifier was trained using search impressions, clicks, average search position, and recent-activity signals available at the prediction snapshot. On the held-out test set, the model achieved an F1 score of 0.663, precision of 0.680, recall of 0.647, and accuracy of 0.819. Model feature importance was dominated by 30-day impressions and recent impression share. The resulting predictions were converted into a ranked list that can support content-review prioritization. The findings demonstrate a potential workflow for using search-performance signals as decision support while highlighting the need for evaluation across additional time periods before operational use.


## 1. Question

*The research question and the decision it supports.*

Research question:
Can search-performance and content-freshness signals be used to identify content items that are likely to be declining, so that content teams can prioritize which items should be reviewed or refreshed?

Decision supported:
The analysis supports a content-prioritization decision: which content items should be reviewed first based on their predicted likelihood of decline.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

The analysis uses the FlyRank internship warehouse release and focuses on anonymized content-performance data. The main source is the daily content-performance fact table, which contains search and traffic performance measures at the client-content-date level.

March 2026 daily performance data was used to construct the March 31, 2026 prediction snapshot. April 2026 daily performance data was used only to construct the future outcome label. The March dataset contains 9,841,378 daily observations, while the April dataset contains 10,424,730 daily observations.

The final modeling population contains 331,436 unique client-content observations present at the March 31 prediction snapshot and having the required April outcome period. An observation is one content item for one client at the prediction snapshot.

The outcome label identifies an item as declining when its April search impressions were more than 20% lower than the preceding 30-day comparison period, provided the preceding period had positive impressions.

The model uses performance signals available by the March 31 snapshot, including impressions, clicks, average search position, pageviews, sessions, users, and recent 7-day activity. Current content metadata with timestamps after the prediction snapshot was excluded from the final feature set to reduce the risk of using information that would not have been available at prediction time.

No client names, private URLs, search queries, or other identifying information are used in the paper outputs.


In [ ]:
%pip -q install duckdb

In [ ]:
%pip -q install duckdb

import duckdb

con = duckdb.connect()

print("DuckDB connected")

DuckDB connected


In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [ ]:
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
""")

print("Hugging Face access configured.")

Hugging Face access configured.


In [ ]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {REL}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_report_date,max_report_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


In [ ]:
columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {REL}
""").df()

columns[["column_name", "column_type"]]

,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


In [ ]:
CONTENT_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
)
"""

content_columns = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {CONTENT_REL}
""").df()

content_columns[["column_name", "column_type"]]

,column_name,column_type
0,client_hash_id,VARCHAR
1,content_hash_id,VARCHAR
2,keyword_hash_id,VARCHAR
3,url_hash_id,VARCHAR
4,keyword_char_count,BIGINT
5,keyword_token_count,BIGINT
6,url_char_count,BIGINT
7,content_created_date,DATE
8,content_updated_date,DATE
9,content_type,VARCHAR


In [ ]:
con.sql(f"""
    SELECT
        report_date,
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS contents
    FROM {REL}
    GROUP BY report_date
    ORDER BY report_date
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,rows,clients,contents
0,2026-03-01,275874,51,275874
1,2026-03-02,276269,51,276269
2,2026-03-03,311676,52,311676
3,2026-03-04,311675,52,311675
4,2026-03-05,311676,52,311676
5,2026-03-06,312187,52,312187
6,2026-03-07,312387,52,312387
7,2026-03-08,313374,52,313374
8,2026-03-09,313874,52,313874
9,2026-03-10,314047,52,314047


In [ ]:
APR_REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
)
"""

con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM {APR_REL}
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_report_date,max_report_date
0,10424730,2026-04-01,2026-04-30


In [ ]:
con.sql(f"""
    SELECT
        COUNT(*) AS april_content_client_pairs
    FROM (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM {APR_REL}
    ) a
    INNER JOIN (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM {REL}
        WHERE report_date = DATE '2026-03-31'
    ) m
    USING (client_hash_id, content_hash_id)
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,april_content_client_pairs
0,331436


In [ ]:
development_labels = con.sql(f"""
    WITH march_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_prev30
        FROM {REL}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),

    april_window AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_current30
        FROM {APR_REL}
        WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.impressions_prev30,
        a.impressions_current30,
        CASE
            WHEN m.impressions_prev30 > 0
             AND a.impressions_current30 < 0.80 * m.impressions_prev30
            THEN 1
            ELSE 0
        END AS is_declining_label
    FROM march_window m
    INNER JOIN april_window a
        USING (client_hash_id, content_hash_id)
""").df()

development_labels.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_prev30,impressions_current30,is_declining_label
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,721.0,249.0,1
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,474.0,281.0,1
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,2839.0,722.0,1
3,client_62f4a7e64f5e0096,content_d47ba5533f9c8573,0.0,0.0,0
4,client_62f4a7e64f5e0096,content_50266f97d6233542,8.0,15.0,0


In [ ]:
development_labels["is_declining_label"].value_counts()


,count
is_declining_label,
0,240262
1,91174


In [ ]:
development_features = con.sql(f"""
    WITH march_30d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            AVG(gsc_avg_position) AS avg_position_30d,
            SUM(ga4_pageviews) AS pageviews_30d,
            SUM(ga4_sessions) AS sessions_30d,
            SUM(ga4_users) AS users_30d
        FROM {REL}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.impressions_30d,
        f.clicks_30d,
        f.avg_position_30d,
        f.pageviews_30d,
        f.sessions_30d,
        f.users_30d,
        d.search_volume,
        d.competition,
        d.keyword_token_count,
        d.word_count,
        d.char_count,
        d.backlinks,
        d.category_count,
        d.content_type,
        d.main_intent,
        d.content_created_date,
        d.content_updated_date,
        d.last_optimized_date,
        d.is_published,
        d.is_deleted
    FROM march_30d f
    INNER JOIN {CONTENT_REL} d
        USING (client_hash_id, content_hash_id)
""").df()

development_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,pageviews_30d,sessions_30d,users_30d,search_volume,competition,...,char_count,backlinks,category_count,content_type,main_intent,content_created_date,content_updated_date,last_optimized_date,is_published,is_deleted
0,client_62f4a7e64f5e0096,content_57fdac1896377109,61.0,0.0,32.332738,NaN,NaN,NaN,10,0.79,...,<NA>,<NA>,0,keyword article,informational,2025-07-25,2026-07-03,NaT,True,False
1,client_62f4a7e64f5e0096,content_96b73204fc32b0cb,839.0,1.0,15.841835,NaN,NaN,NaN,10,0.00,...,16354,<NA>,0,keyword article,transactional,2025-07-25,2026-07-04,2026-07-04,True,False
2,client_62f4a7e64f5e0096,content_4a6c7ca3319158fd,3660.0,7.0,5.247409,NaN,NaN,NaN,10,0.00,...,16027,<NA>,0,keyword article,transactional,2025-07-25,2026-07-03,2026-06-24,True,False
3,client_62f4a7e64f5e0096,content_0cd03ae2d0316650,19.0,0.0,69.535714,NaN,NaN,NaN,10,0.43,...,<NA>,<NA>,0,keyword article,commercial,2025-07-25,2026-07-03,NaT,True,False
4,client_62f4a7e64f5e0096,content_c505daaa3ffc8e6e,1668.0,13.0,4.416615,NaN,NaN,NaN,10,0.00,...,15870,<NA>,0,keyword article,transactional,2025-07-25,2026-07-06,2026-07-06,True,False


In [ ]:
development_features.shape

(331436, 22)

In [ ]:
development_features.isna().sum()

,0
client_hash_id,0
content_hash_id,0
impressions_30d,0
clicks_30d,0
avg_position_30d,155168
pageviews_30d,70700
sessions_30d,70700
users_30d,70700
search_volume,59406
competition,59406


In [ ]:
development_features = development_features.copy()

# Missingness indicators preserve information about whether a value was recorded.
development_features["has_optimization_date"] = (
    development_features["last_optimized_date"].notna().astype(int)
)

development_features["has_word_count"] = (
    development_features["word_count"].notna().astype(int)
)

development_features["has_backlinks"] = (
    development_features["backlinks"].notna().astype(int)
)

development_features["has_search_volume"] = (
    development_features["search_volume"].notna().astype(int)
)

# Numeric missing values are filled with 0 for modeling.
# The missingness indicators above help the model distinguish
# a true zero from an originally missing value.
numeric_fill_zero = [
    "pageviews_30d",
    "sessions_30d",
    "users_30d",
    "search_volume",
    "competition",
    "word_count",
    "char_count",
    "backlinks",
    "avg_position_30d",
]

for col in numeric_fill_zero:
    development_features[col] = development_features[col].fillna(0)

development_features.head()

,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,pageviews_30d,sessions_30d,users_30d,search_volume,competition,...,main_intent,content_created_date,content_updated_date,last_optimized_date,is_published,is_deleted,has_optimization_date,has_word_count,has_backlinks,has_search_volume
0,client_62f4a7e64f5e0096,content_57fdac1896377109,61.0,0.0,32.332738,0.0,0.0,0.0,10,0.79,...,informational,2025-07-25,2026-07-03,NaT,True,False,0,0,0,1
1,client_62f4a7e64f5e0096,content_96b73204fc32b0cb,839.0,1.0,15.841835,0.0,0.0,0.0,10,0.00,...,transactional,2025-07-25,2026-07-04,2026-07-04,True,False,1,1,0,1
2,client_62f4a7e64f5e0096,content_4a6c7ca3319158fd,3660.0,7.0,5.247409,0.0,0.0,0.0,10,0.00,...,transactional,2025-07-25,2026-07-03,2026-06-24,True,False,1,1,0,1
3,client_62f4a7e64f5e0096,content_0cd03ae2d0316650,19.0,0.0,69.535714,0.0,0.0,0.0,10,0.43,...,commercial,2025-07-25,2026-07-03,NaT,True,False,0,0,0,1
4,client_62f4a7e64f5e0096,content_c505daaa3ffc8e6e,1668.0,13.0,4.416615,0.0,0.0,0.0,10,0.00,...,transactional,2025-07-25,2026-07-06,2026-07-06,True,False,1,1,0,1


In [ ]:
development_features.isna().sum()

,0
client_hash_id,0
content_hash_id,0
impressions_30d,0
clicks_30d,0
avg_position_30d,0
pageviews_30d,0
sessions_30d,0
users_30d,0
search_volume,0
competition,0


In [ ]:
metadata_date_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE content_updated_date > DATE '2026-03-31'
        ) AS updated_after_snapshot,
        COUNT(*) FILTER (
            WHERE last_optimized_date > DATE '2026-03-31'
        ) AS optimized_after_snapshot,
        COUNT(*) FILTER (
            WHERE content_created_date > DATE '2026-03-31'
        ) AS created_after_snapshot
    FROM {CONTENT_REL}
""").df()

metadata_date_check

,total_rows,updated_after_snapshot,optimized_after_snapshot,created_after_snapshot
0,519606,382739,45396,86172


In [ ]:
metadata_date_check

,total_rows,updated_after_snapshot,optimized_after_snapshot,created_after_snapshot
0,519606,382739,45396,86172


In [ ]:
joined_date_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE d.content_created_date > DATE '2026-03-31'
        ) AS created_after_snapshot,
        COUNT(*) FILTER (
            WHERE d.content_updated_date > DATE '2026-03-31'
        ) AS updated_after_snapshot,
        COUNT(*) FILTER (
            WHERE d.last_optimized_date > DATE '2026-03-31'
        ) AS optimized_after_snapshot
    FROM (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM {REL}
        WHERE report_date = DATE '2026-03-31'
    ) m
    INNER JOIN {CONTENT_REL} d
        USING (client_hash_id, content_hash_id)
""").df()

joined_date_check

,total_rows,created_after_snapshot,updated_after_snapshot,optimized_after_snapshot
0,331436,2124,293357,42517


In [ ]:
development_features_safe = con.sql(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_pageviews,
            ga4_sessions,
            ga4_users
        FROM {REL}
    ),

    march_summary AS (
        SELECT
            client_hash_id,
            content_hash_id,

            -- Full 30-day performance
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            AVG(gsc_avg_position) AS avg_position_30d,

            SUM(ga4_pageviews) AS pageviews_30d,
            SUM(ga4_sessions) AS sessions_30d,
            SUM(ga4_users) AS users_30d,

            -- Recent 7-day performance
            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_7d,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clicks_7d,

            AVG(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN gsc_avg_position
                END
            ) AS avg_position_7d

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        *,

        -- Simple recent-vs-full-period performance signals
        CASE
            WHEN impressions_30d > 0
            THEN impressions_7d / impressions_30d
            ELSE 0
        END AS impressions_recent_share,

        CASE
            WHEN clicks_30d > 0
            THEN clicks_7d / clicks_30d
            ELSE 0
        END AS clicks_recent_share

    FROM march_summary
""").df()

development_features_safe.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,pageviews_30d,sessions_30d,users_30d,impressions_7d,clicks_7d,avg_position_7d,impressions_recent_share,clicks_recent_share
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,1.0,1.0,865.0,1.0,6.060857,0.132608,0.142857
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,0.0,0.0,93.0,0.0,2.401410,0.205298,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,3.0,3.0,726.0,2.0,7.566592,0.128952,0.333333
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,2.0,2.0,2.0,803.0,1.0,6.393425,0.162419,0.076923
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,8.0,7.0,6.0,10.0,0.0,29.150000,0.238095,0.000000


In [ ]:
development_features_safe.shape

(331437, 13)

In [ ]:
development_features_safe.isna().sum()

,0
client_hash_id,0
content_hash_id,0
impressions_30d,0
clicks_30d,0
avg_position_30d,154699
pageviews_30d,70700
sessions_30d,70700
users_30d,70700
impressions_7d,0
clicks_7d,0


In [ ]:
development_features_safe = con.sql(f"""
    WITH march_30d AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions_30d,
            SUM(gsc_clicks) AS clicks_30d,
            AVG(gsc_avg_position) AS avg_position_30d,
            SUM(ga4_pageviews) AS pageviews_30d,
            SUM(ga4_sessions) AS sessions_30d,
            SUM(ga4_users) AS users_30d,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS impressions_7d,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clicks_7d,

            AVG(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-25'
                                         AND DATE '2026-03-31'
                    THEN gsc_avg_position
                END
            ) AS avg_position_7d

        FROM {REL}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
        GROUP BY client_hash_id, content_hash_id
    ),

    march_31_pairs AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM {REL}
        WHERE report_date = DATE '2026-03-31'
    )

    SELECT
        m.*,

        CASE
            WHEN m.impressions_30d > 0
            THEN m.impressions_7d / m.impressions_30d
            ELSE 0
        END AS impressions_recent_share,

        CASE
            WHEN m.clicks_30d > 0
            THEN m.clicks_7d / m.clicks_30d
            ELSE 0
        END AS clicks_recent_share

    FROM march_30d m
    INNER JOIN march_31_pairs p
        USING (client_hash_id, content_hash_id)
""").df()

print("Feature rows:", len(development_features_safe))
print("Feature columns:", len(development_features_safe.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331436
Feature columns: 13


In [ ]:
development_features_safe.shape

(331436, 13)

In [ ]:
development_features_safe.isna().sum()

,0
client_hash_id,0
content_hash_id,0
impressions_30d,0
clicks_30d,0
avg_position_30d,154699
pageviews_30d,70700
sessions_30d,70700
users_30d,70700
impressions_7d,0
clicks_7d,0


In [ ]:
model_data = development_features_safe.merge(
    development_labels[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_prev30",
            "impressions_current30",
            "is_declining_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

print("Model rows:", len(model_data))
print("Model columns:", len(model_data.columns))

Model rows: 331436
Model columns: 16


In [ ]:
model_data["is_declining_label"].value_counts()

,count
is_declining_label,
0,240262
1,91174


In [ ]:
duplicate_pairs = (
    model_data
    .groupby(["client_hash_id", "content_hash_id"])
    .size()
    .reset_index(name="n")
)

duplicate_pairs[duplicate_pairs["n"] > 1]

,client_hash_id,content_hash_id,n


In [ ]:
model_data = model_data.copy()

# GA4 activity: missing values are treated as zero observed activity.
for col in ["pageviews_30d", "sessions_30d", "users_30d"]:
    model_data[col] = model_data[col].fillna(0)

# Search position: missing means no usable position observation.
# Use the median of observed values rather than treating it as position 0.
for col in ["avg_position_30d", "avg_position_7d"]:
    median_value = model_data[col].median()
    model_data[col] = model_data[col].fillna(median_value)

print("Missing values remaining:")
print(model_data.isna().sum())

Missing values remaining:
client_hash_id              0
content_hash_id             0
impressions_30d             0
clicks_30d                  0
avg_position_30d            0
pageviews_30d               0
sessions_30d                0
users_30d                   0
impressions_7d              0
clicks_7d                   0
avg_position_7d             0
impressions_recent_share    0
clicks_recent_share         0
impressions_prev30          0
impressions_current30       0
is_declining_label          0
dtype: int64


In [ ]:
model_data[
    [
        "impressions_30d",
        "clicks_30d",
        "avg_position_30d",
        "pageviews_30d",
        "sessions_30d",
        "users_30d",
        "impressions_7d",
        "clicks_7d",
        "avg_position_7d",
        "impressions_recent_share",
        "clicks_recent_share"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
impressions_30d,331436.0,846.792708,4044.520587,0.0,0.000000,2.000000,216.000000,617124.0
clicks_30d,331436.0,2.479610,19.651311,0.0,0.000000,0.000000,0.000000,5668.0
avg_position_30d,331436.0,12.501527,13.445419,0.0,8.000000,8.505445,9.150006,309.0
pageviews_30d,331436.0,4.480069,28.321729,0.0,0.000000,0.000000,1.000000,2879.0
sessions_30d,331436.0,3.921747,25.381920,0.0,0.000000,0.000000,1.000000,2730.0
users_30d,331436.0,3.830676,25.005641,0.0,0.000000,0.000000,1.000000,2648.0
impressions_7d,331436.0,217.016250,1195.550662,0.0,0.000000,0.000000,49.000000,245414.0
clicks_7d,331436.0,0.578854,5.250624,0.0,0.000000,0.000000,0.000000,1618.0
avg_position_7d,331436.0,12.023101,13.386476,0.0,8.246778,8.246778,8.246778,305.5
impressions_recent_share,331436.0,0.150533,0.230797,0.0,0.000000,0.000000,0.237288,1.0


In [ ]:
feature_columns = [
    "impressions_30d",
    "clicks_30d",
    "avg_position_30d",
    "pageviews_30d",
    "sessions_30d",
    "users_30d",
    "impressions_7d",
    "clicks_7d",
    "avg_position_7d",
    "impressions_recent_share",
    "clicks_recent_share"
]

X = model_data[feature_columns].copy()
y = model_data["is_declining_label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Positive labels:", int(y.sum()))
print("Negative labels:", int((y == 0).sum()))

X shape: (331436, 11)
y shape: (331436,)
Positive labels: 91174
Negative labels: 240262


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training declining rate:", round(y_train.mean(), 4))
print("Test declining rate:", round(y_test.mean(), 4))

Training rows: 265148
Test rows: 66288
Training declining rate: 0.2751
Test declining rate: 0.2751


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Modeling approach

The analysis uses a supervised binary classification approach. Each observation represents one content item for one client at the March 31, 2026 prediction snapshot. The target variable indicates whether the item's search impressions declined by more than 20% in the following 30-day period.

The model uses search-performance and recent-activity signals available by March 31, including 30-day and 7-day impressions and clicks, average search position, and recent shares of impressions and clicks. April performance is used only to construct the outcome label and is not used as a model feature.

The development dataset contains 331,436 observations. It was divided into an 80% training set and a 20% test set using stratified sampling, preserving the declining rate of 27.51% in both sets. A Decision Tree classifier is used as the initial predictive model because it can capture nonlinear relationships and is relatively easy to interpret for a content-prioritization use case.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [ ]:
y_pred = model.predict(X_test)

print("Predictions generated:", len(y_pred))

Predictions generated: 66288


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall   :", round(recall, 3))
print("F1 score :", round(f1, 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.819
Precision: 0.68
Recall   : 0.647
F1 score : 0.663

Confusion matrix:
[[42506  5547]
 [ 6441 11794]]


In [ ]:
results_comparison = pd.DataFrame({
    "Model": ["Baseline", "Decision Tree"],
    "Accuracy": [None, accuracy_score(y_test, y_pred)],
    "Precision": [0.495, precision_score(y_test, y_pred)],
    "Recall": [0.533, recall_score(y_test, y_pred)],
    "F1": [0.513, f1_score(y_test, y_pred)]
})

results_comparison

,Model,Accuracy,Precision,Recall,F1
0,Baseline,NaN,0.495000,0.533000,0.513000
1,Decision Tree,0.819153,0.680122,0.646778,0.663031


In [ ]:
test_recommendations = X_test.copy()

test_recommendations["actual_declining"] = y_test.values
test_recommendations["predicted_declining"] = y_pred
test_recommendations["decline_probability"] = model.predict_proba(X_test)[:, 1]

test_recommendations = (
    test_recommendations
    .sort_values("decline_probability", ascending=False)
    .reset_index(drop=True)
)

test_recommendations.head(10)

,impressions_30d,clicks_30d,avg_position_30d,pageviews_30d,sessions_30d,users_30d,impressions_7d,clicks_7d,avg_position_7d,impressions_recent_share,clicks_recent_share,actual_declining,predicted_declining,decline_probability
0,339.0,0.0,7.757313,0.0,0.0,0.0,0.0,0.0,8.246778,0.000000,0.0,1,1,0.840522
1,440.0,1.0,14.374745,0.0,0.0,0.0,23.0,0.0,10.035714,0.052273,0.0,1,1,0.840522
2,109.0,0.0,76.073524,0.0,0.0,0.0,13.0,0.0,75.625000,0.119266,0.0,1,1,0.840522
3,1395.0,2.0,2.874113,3.0,3.0,3.0,68.0,1.0,5.539510,0.048746,0.5,1,1,0.840522
4,83.0,0.0,9.656097,0.0,0.0,0.0,0.0,0.0,8.246778,0.000000,0.0,1,1,0.840522
5,883.0,1.0,18.527633,0.0,0.0,0.0,76.0,0.0,24.146259,0.086070,0.0,1,1,0.840522
6,290.0,0.0,21.734291,0.0,0.0,0.0,20.0,0.0,24.372917,0.068966,0.0,1,1,0.840522
7,2222.0,1.0,39.749531,44.0,44.0,45.0,239.0,0.0,27.748161,0.107561,0.0,0,1,0.840522
8,420.0,0.0,5.405139,0.0,0.0,0.0,23.0,0.0,6.059524,0.054762,0.0,1,1,0.840522
9,76.0,0.0,54.141567,0.0,0.0,0.0,9.0,0.0,39.000000,0.118421,0.0,1,1,0.840522


In [ ]:
ranked_recommendations = X_test.copy()

ranked_recommendations["client_hash_id"] = model_data.loc[
    X_test.index, "client_hash_id"
].values

ranked_recommendations["content_hash_id"] = model_data.loc[
    X_test.index, "content_hash_id"
].values

ranked_recommendations["actual_declining"] = y_test.values

ranked_recommendations["decline_probability"] = model.predict_proba(X_test)[:, 1]

ranked_recommendations = (
    ranked_recommendations
    .sort_values(
        ["decline_probability", "impressions_30d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

ranked_recommendations[
    [
        "client_hash_id",
        "content_hash_id",
        "decline_probability",
        "actual_declining",
        "impressions_30d",
        "impressions_recent_share"
    ]
].head(10)

,client_hash_id,content_hash_id,decline_probability,actual_declining,impressions_30d,impressions_recent_share
0,client_23a62021009f63c4,content_573804af4f4fa09f,0.840522,1,74086.0,0.116621
1,client_23a62021009f63c4,content_1ae5eb3539e7ad9e,0.840522,1,61332.0,0.103698
2,client_62f4a7e64f5e0096,content_bb2a9972810ddd72,0.840522,1,53805.0,0.050553
3,client_23a62021009f63c4,content_b77c09e114e0ed4c,0.840522,1,47736.0,0.114672
4,client_23a62021009f63c4,content_f8c0566c8f017176,0.840522,1,43138.0,0.107794
5,client_62f4a7e64f5e0096,content_0c5606abaaab3178,0.840522,1,38865.0,0.003371
6,client_23a62021009f63c4,content_a12d89af10c7513a,0.840522,1,36535.0,0.125414
7,client_23a62021009f63c4,content_e70886c63f95aa1c,0.840522,1,32715.0,0.120312
8,client_62f4a7e64f5e0096,content_48cb9501333b94af,0.840522,1,29837.0,0.110232
9,client_23a62021009f63c4,content_3455e711fb9ed498,0.840522,1,29822.0,0.109919


In [ ]:
top10_summary = {
    "top_10_count": len(ranked_recommendations.head(10)),
    "top_10_declining": int(ranked_recommendations.head(10)["actual_declining"].sum()),
    "top_10_decline_rate": ranked_recommendations.head(10)["actual_declining"].mean(),
    "top_probability": ranked_recommendations.head(10)["decline_probability"].iloc[0],
    "highest_impressions_30d": ranked_recommendations.head(10)["impressions_30d"].max(),
    "lowest_impressions_30d": ranked_recommendations.head(10)["impressions_30d"].min()
}

top10_summary

{'top_10_count': 10,
 'top_10_declining': 10,
 'top_10_decline_rate': np.float64(1.0),
 'top_probability': np.float64(0.8405222437137331),
 'highest_impressions_30d': 74086.0,
 'lowest_impressions_30d': 29822.0}

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The Decision Tree achieved an F1 score of 0.663 on the held-out test set, compared with 0.513 for the earlier baseline. Precision increased from 0.495 for the baseline to 0.680 for the Decision Tree, while recall increased from 0.533 to 0.647.

The Decision Tree achieved an accuracy of 0.819 on the held-out test set. The confusion matrix contained 42,506 true negatives, 5,547 false positives, 6,441 false negatives, and 11,794 true positives.

Feature importance showed that `impressions_30d` was the main feature used by the tree, with an importance of 0.775, followed by `impressions_recent_share` at 0.200. The remaining features had substantially smaller importance values. This indicates that the model's predictions were driven mainly by recent search-impression signals.

The ranked recommendation output also provided a practical prioritization mechanism. The ten highest-ranked observations in the held-out test set all had an observed declining label, although this result applies only to these ten test observations and should not be interpreted as a guarantee for future recommendations.


## 5. Limitations

*What this work cannot claim.*

This analysis has several limitations.

First, the model was developed using a March 2026 prediction snapshot and evaluated against the following April 2026 outcome period. This provides a time-based development and evaluation setup, but it does not show how the model would perform across many different future periods.

Second, the target is based on a specific definition of decline: search impressions falling by more than 20% compared with the previous 30-day period. Other definitions of content decline could produce different labels and model results.

Third, the available data contains missing values. Some search-position and GA4 traffic fields were unavailable for a substantial number of observations. Missing GA4 traffic values were treated as zero, while missing search-position values were filled using the median. These choices may affect model performance.

Fourth, current content metadata was not used as a modeling feature when its timestamps showed that the information could have been updated after the March 31 prediction snapshot. This was done to reduce the risk of using information that would not have been available at prediction time, but it also limits the range of content-level signals included in the model.

Fifth, the Decision Tree relies heavily on two impression-based features: `impressions_30d` and `impressions_recent_share`. This means the model's predictions are strongly influenced by search-impression patterns and may not capture other reasons why content declines, such as changes in search intent, competition, content quality, technical issues, or external events.

Finally, the ranked recommendations were evaluated on the held-out test set. The observation that all ten of the highest-ranked test items had an observed decline should therefore be treated as a result for this test sample, not as evidence that future recommendations will have the same success rate. Further evaluation on later time periods would be needed before using the model operationally.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The model can be used to prioritize content items for review by ranking them according to their predicted probability of decline. This supports a review-first workflow in which content teams can focus attention on items that the model identifies as having a higher likelihood of declining.

For the held-out test set, the ten highest-ranked observations all received a predicted decline probability of approximately 0.841. Because the Decision Tree assigned the same probability to these observations, `impressions_30d` was used as a transparent tie-breaker, placing observations with higher recent search-impression volume first.

The top ten ranked observations are shown below:

| Rank | Client ID               | Content ID               | Predicted decline probability | Observed decline | Impressions (30d) |
| ---: | ----------------------- | ------------------------ | ----------------------------: | ---------------: | ----------------: |
|    1 | client_23a62021009f63c4 | content_573804af4f4fa09f |                         0.841 |                1 |            74,086 |
|    2 | client_23a62021009f63c4 | content_1ae5eb3539e7ad9e |                         0.841 |                1 |            61,332 |
|    3 | client_62f4a7e64f5e0096 | content_bb2a9972810ddd72 |                         0.841 |                1 |            53,805 |
|    4 | client_23a62021009f63c4 | content_b77c09e114e0ed4c |                         0.841 |                1 |            47,736 |
|    5 | client_23a62021009f63c4 | content_f8c0566c8f017176 |                         0.841 |                1 |            43,138 |
|    6 | client_62f4a7e64f5e0096 | content_0c5606abaaab3178 |                         0.841 |                1 |            38,865 |
|    7 | client_23a62021009f63c4 | content_a12d89af10c7513a |                         0.841 |                1 |            36,535 |
|    8 | client_23a62021009f63c4 | content_e70886c63f95aa1c |                         0.841 |                1 |            32,715 |
|    9 | client_62f4a7e64f5e0096 | content_48cb9501333b94af |                         0.841 |                1 |            29,837 |
|   10 | client_23a62021009f63c4 | content_3455e711fb9ed498 |                         0.841 |                1 |            29,822 |

All ten observations in this top-ten test-set ranking had an observed declining label. Their 30-day impression volume ranged from 29,822 to 74,086 impressions.

These rankings should be interpreted as a prioritization aid rather than an automatic refresh decision. A content team could use the ranked list to identify items for further review and then consider additional information, such as content quality, search intent, technical issues, and business priorities, before deciding whether an item should actually be refreshed.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The analysis is supported by three main artifacts.

1. `capstone_model_results.csv` contains the evaluation metrics for the baseline and Decision Tree model, including precision, recall, F1 score, and accuracy where available.

2. `capstone_feature_importance.csv` contains the Decision Tree feature-importance values used to identify which available signals contributed most to the model's predictions.

3. `capstone_ranked_recommendations.csv` contains the top ten anonymized content observations ranked by predicted probability of decline. The file includes the anonymized client and content identifiers, predicted decline probability, observed outcome, 30-day impressions, and recent impression share.

These artifacts provide reproducible supporting evidence for the model evaluation, feature analysis, and content-prioritization results presented in the paper.


In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

# 1. Model evaluation results
results_comparison.to_csv(
    "work/outputs/capstone_model_results.csv",
    index=False
)

# 2. Feature importance
feature_importance.to_csv(
    "work/outputs/capstone_feature_importance.csv",
    index=False
)

# 3. Ranked recommendations
ranked_recommendations[
    [
        "client_hash_id",
        "content_hash_id",
        "decline_probability",
        "actual_declining",
        "impressions_30d",
        "impressions_recent_share"
    ]
].head(10).to_csv(
    "work/outputs/capstone_ranked_recommendations.csv",
    index=False
)

print("Capstone artifacts saved successfully.")

Capstone artifacts saved successfully.


In [ ]:
import os

artifact_paths = [
    "work/outputs/capstone_model_results.csv",
    "work/outputs/capstone_feature_importance.csv",
    "work/outputs/capstone_ranked_recommendations.csv"
]

for path in artifact_paths:
    print(path, "->", os.path.exists(path))

work/outputs/capstone_model_results.csv -> True
work/outputs/capstone_feature_importance.csv -> True
work/outputs/capstone_ranked_recommendations.csv -> True


## ML-12 Closing Deliverables

### 5-Minute Demo Outline

**0:00–0:30 — Problem and decision**

Introduce the research question: whether search-performance and recent-activity signals can identify content items that are likely to decline, so that content teams can prioritize which items to review.

**0:30–1:30 — Data and prediction setup**

Explain that the analysis uses anonymized content-performance data. March 31, 2026 is used as the prediction snapshot, while April 2026 provides the future outcome used to define decline. The final development dataset contains 331,436 observations.

**1:30–2:30 — Features and model**

Explain that the model uses search-performance signals such as 30-day and 7-day impressions and clicks, average search position, and recent shares of impressions and clicks. A Decision Tree classifier was trained using an 80/20 stratified train-test split.

**2:30–3:30 — Results**

Show the model results: F1 of 0.663, precision of 0.680, recall of 0.647, and accuracy of 0.819 on the held-out test set. Compare these with the earlier baseline F1 of 0.513, precision of 0.495, and recall of 0.533.

**3:30–4:15 — What the model used**

Show the feature-importance results. The largest importance values were `impressions_30d` at 0.775 and `impressions_recent_share` at 0.200. Explain that the model therefore relied mainly on recent search-impression patterns.

**4:15–5:00 — Recommendation and limitations**

Show the ranked recommendations and explain that the model can help prioritize content for human review rather than automatically deciding which content should be refreshed. Close by noting that evaluation across additional future periods would be needed before operational use.

### Social-Post Cut

Built a supervised machine-learning workflow to identify content items that may be experiencing search-performance decline. Using 331,436 anonymized content-client observations, a Decision Tree achieved an F1 score of 0.663 on a held-out test set. The resulting probability ranking provides a practical way to prioritize content for human review while keeping the final refresh decision with the content team.

### Employer-Facing Summary

I developed a supervised classification workflow to identify content items with a higher likelihood of search-performance decline. Using 331,436 anonymized observations and a Decision Tree model, the workflow achieved an F1 score of 0.663, with predictions driven mainly by recent search-impression signals. I also converted the model predictions into a ranked recommendation output that can support content-review prioritization while accounting for the limitations of a single evaluation period.


In [ ]:
# Capstone self-check

checks = {
    "Research question defined": True,
    "Decision supported defined": True,
    "Development observations": len(model_data) == 331436,
    "Positive labels": int(y.sum()) == 91174,
    "Train/test split completed": len(X_train) == 265148 and len(X_test) == 66288,
    "No label-derived features used": (
        "trend_direction" not in feature_columns
        and "trend_pct" not in feature_columns
        and "is_declining_label" not in feature_columns
    ),
    "Model trained": hasattr(model, "feature_importances_"),
    "Test predictions generated": len(y_pred) == len(y_test),
    "F1 calculated": True,
    "Ranked recommendations generated": len(ranked_recommendations) == len(X_test),
    "Top 10 recommendations generated": len(ranked_recommendations.head(10)) == 10,
    "Model results artifact exists": os.path.exists(
        "work/outputs/capstone_model_results.csv"
    ),
    "Feature importance artifact exists": os.path.exists(
        "work/outputs/capstone_feature_importance.csv"
    ),
    "Recommendations artifact exists": os.path.exists(
        "work/outputs/capstone_ranked_recommendations.csv"
    )
}

for check, passed in checks.items():
    print(("PASS" if passed else "FAIL") + " - " + check)

print("\nAll checks passed:", all(checks.values()))

PASS - Research question defined
PASS - Decision supported defined
PASS - Development observations
PASS - Positive labels
PASS - Train/test split completed
PASS - No label-derived features used
PASS - Model trained
PASS - Test predictions generated
PASS - F1 calculated
PASS - Ranked recommendations generated
PASS - Top 10 recommendations generated
PASS - Model results artifact exists
PASS - Feature importance artifact exists
PASS - Recommendations artifact exists

All checks passed: True


This capstone developed a supervised machine-learning workflow for prioritizing content items that may be experiencing search-performance decline.

The analysis used March 31, 2026 as the prediction snapshot and evaluated the subsequent April 2026 outcome period. The final development dataset contained 331,436 content-client observations. A Decision Tree classifier was trained using search-performance and recent-activity signals available at the prediction snapshot.

On the held-out test set, the Decision Tree achieved an F1 score of 0.663, with precision of 0.680, recall of 0.647, and accuracy of 0.819. The model relied most heavily on 30-day impressions and the recent share of impressions.

The resulting probability ranking provides a practical way to prioritize content for human review. The model should be treated as a decision-support tool rather than an automatic content-refresh system. Further testing across additional time periods would be required before operational deployment.


## Acknowledgments & Data Credit

This work was completed as part of the FlyRank AI internship and uses the anonymized internship warehouse dataset provided for the program.

The analysis was performed using anonymized content, client, and search-performance identifiers. No client names, private URLs, search queries, or other identifying information are included in the analysis outputs.

The dataset and analysis are used for educational and internship purposes. The findings represent the analysis performed in this capstone and should not be interpreted as a production deployment or as a guarantee of future content performance.

Data and program credit: https://flyrank.ai


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
